# Analisi delle Serie Storiche e Pipeline a Due Stadi (Two-Stage Forecasting): Kabul

Questo notebook contiene l'analisi dettagliata per la provincia di **Kabul (AF01)**:
1. **Allineamento Multigranularità**: Come allineare dati WFP, Rainfall, ACLED, IDP e IPC a una frequenza mensile coerente.
2. **Stage 1 (Univariate Forecast)**: Previsione univariata (Holt-Winters / SARIMAX) per ciascuno dei 9 predittori ambientali e socio-economici a 12 mesi nel futuro.
3. **Stage 2 (Multivariate Project)**: Addestramento del modello Random Forest Regressor e proiezione dell'indice di insicurezza alimentare **IPC Phase 3+ %** futuro.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
os.makedirs("plots", exist_ok=True)

In [ ]:
base = "../../data"
wfp = pd.read_parquet(f"{base}/tmp/wfp_monthly_adm1_index.parquet", engine="pyarrow")
rain = pd.read_parquet(f"{base}/raw/rainfall.parquet", engine="pyarrow")
acled = pd.read_parquet(f"{base}/raw/acled.parquet", engine="pyarrow")
idp = pd.read_parquet(f"{base}/raw/idp.parquet", engine="pyarrow")
merged = pd.read_parquet(f"{base}/merged/merged_adm1_wide.parquet", engine="pyarrow")

# 1. WFP Kabul
wfp_k = wfp[(wfp["ISO3"] == "AFG") & (wfp["adm1_pcode"] == "AF01")].copy()
wfp_k["date"] = pd.to_datetime(wfp_k["date"])
wfp_k = wfp_k.set_index("date").sort_index()[["wfp_price_mean", "wfp_inflation_mean"]].asfreq("MS")
wfp_k = wfp_k.interpolate(method="linear")

# 2. Rainfall Kabul
rain_k = rain[rain["PCODE"] == "AF01"].copy()
rain_k["date"] = pd.to_datetime(rain_k["date"]).dt.to_period("M").dt.to_timestamp()
rain_k = rain_k.groupby("date")[["rain_1m", "rain_3m", "rain_anomaly_1m", "rain_anomaly_3m"]].mean().asfreq("MS")

# 3. ACLED Kabul
acled_k = acled[acled["admin1_code"] == "AF01"].copy()
acled_k["date"] = pd.to_datetime(acled_k["reference_period_start"]).dt.to_period("M").dt.to_timestamp()
acled_k = acled_k.groupby("date")[["events", "fatalities"]].sum().rename(
    columns={"events": "acled_events", "fatalities": "acled_fatalities"}
).asfreq("MS", fill_value=0)

# 4. IDP Kabul
idp_k = idp[idp["admin1_code"] == "AF01"].copy()
idp_k["date"] = pd.to_datetime(idp_k["reference_period_start"]).dt.to_period("M").dt.to_timestamp()
idp_k = idp_k.groupby("date")[["population"]].mean().rename(
    columns={"population": "idp_population"}
).asfreq("MS").ffill().fillna(0)

# 5. IPC Kabul
ipc_k = merged[(merged["adm1_pcode"] == "AF01") & (merged["Validity period"] == "current")].copy()
expanded = []
for _, row in ipc_k.iterrows():
    m_range = pd.date_range(start=row["From"], end=row["To"], freq="MS")
    for m in m_range:
        expanded.append({"date": m, "ipc_phase_3plus_pct": row["phase_3plus_percentage"]})
df_ipc = pd.DataFrame(expanded).drop_duplicates(subset=["date"]).set_index("date").sort_index().asfreq("MS")
df_ipc_filled = df_ipc.ffill().bfill()

# Join
joined = wfp_k.join([rain_k, acled_k, idp_k, df_ipc_filled], how="inner")
print(f"Joined Shape: {joined.shape}")
print(joined.head(5))

### Stage 1: Univariate Forecasting dei Predittori
Prevediamo in avanti di 12 mesi ciascuna variabile indipendente per poter simulare lo scenario futuro di Kabul.

In [ ]:
predictors = [
    "wfp_price_mean", "wfp_inflation_mean", 
    "rain_1m", "rain_3m", "rain_anomaly_1m", "rain_anomaly_3m", 
    "acled_events", "acled_fatalities", 
    "idp_population"
]

forecast_steps = 12
forecast_index = pd.date_range(start=joined.index[-1] + pd.DateOffset(months=1), periods=forecast_steps, freq="MS")
forecasted_predictors = pd.DataFrame(index=forecast_index)

for col in predictors:
    ts = joined[col]
    try:
        if col in ["acled_events", "acled_fatalities", "idp_population"]:
            model = ExponentialSmoothing(ts, trend="add", seasonal=None)
        else:
            model = ExponentialSmoothing(ts, trend="add", seasonal="add", seasonal_periods=12)
        res = model.fit()
        fc = res.forecast(steps=forecast_steps)
    except Exception:
        model = SARIMAX(ts, order=(1, 1, 1), enforce_stationarity=False, enforce_invertibility=False)
        res = model.fit(disp=False)
        fc = res.forecast(steps=forecast_steps)
        
    # Vincolo non-negatività
    if col in ["rain_1m", "rain_3m", "acled_events", "acled_fatalities", "idp_population", "wfp_price_mean"]:
        fc = fc.clip(lower=0)
        
    forecasted_predictors[col] = fc
    
    # Plot
    plt.figure(figsize=(12, 4))
    plt.plot(ts.loc["2020-01-01":], label="Storico Reale", color="black", linewidth=2)
    plt.plot(fc, label="Forecast Future (12m)", color="red", linestyle="--", marker='o')
    plt.axvline(x=ts.index[-1], color="gray", linestyle=":")
    plt.title(f"Univariate Forecast: {col} - Kabul")
    plt.legend(loc="upper left")
    plt.savefig(f"plots/univariate_{col}.png", dpi=150, bbox_inches="tight")
    plt.show()

### Stage 2: Proiezione Multivariata dell'IPC
Addestriamo il Random Forest Regressor e utilizziamo le feature stimate dallo Stage 1 per proiettare l'andamento futuro dell'insicurezza alimentare a Kabul.

In [ ]:
X = joined[predictors]
y = joined["ipc_phase_3plus_pct"]

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Predizioni storiche e future
joined["predicted_ipc"] = rf.predict(X)
future_pred = rf.predict(forecasted_predictors[predictors])
df_future = pd.DataFrame(index=forecast_index)
df_future["predicted_ipc"] = future_pred

# Plot Finale
plt.figure(figsize=(14, 6))
plt.plot(joined.index, joined["ipc_phase_3plus_pct"], label="Actual IPC Phase 3+ %", color="black", marker="o", linewidth=2.5)
plt.plot(joined.index, joined["predicted_ipc"], label="Fitted IPC % (Random Forest)", color="blue", linestyle="--", linewidth=1.5)
plt.plot(df_future.index, df_future["predicted_ipc"], label="Projected Future IPC % (12m ahead)", color="red", linestyle="-.", marker="s", linewidth=2)
plt.axvline(x=joined.index[-1], color="gray", linestyle=":", linewidth=2, label="Inizio Forecast")
plt.title("Proiezione dell'Indice IPC Phase 3+ % per Kabul (Two-Stage Forecasting)", fontsize=14)
plt.xlabel("Data")
plt.ylabel("% Popolazione in Phase 3+")
plt.legend(loc="upper left")
plt.savefig("plots/kabul_multivariate_ipc_forecast.png", dpi=150, bbox_inches="tight")
plt.show()